# RAG at Scale — Complete Demo

**How to run (pick one):**

| Method | Command | Kernel to select |
|---|---|---|
| **Local** (recommended now) | Open this notebook in VS Code / Jupyter | `rag-at-scale` (Python 3.11 from this repo's `venv`) |
| **Docker** | `run_demo.bat` → open http://localhost:8888?token=ragdemo | pre-selected inside container |

**Topics:** Spark · Distributed Embeddings · Hybrid Search · Rerank · FastAPI · Observability · AWS Bedrock · Docker Deploy

**Story hook:** the demo corpus mixes one real RAG trend brief with several fictional failure incidents, so students can clearly see when chunking, hybrid retrieval, and reranking recover the right narrative.

---

In [1]:
import sys, os, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd

# Tell Spark's JVM to use THIS Python for worker processes
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Support both Docker (/home/jovyan/work) and local execution (cwd)
_docker_root = Path('/home/jovyan/work')
PROJECT_ROOT = _docker_root if _docker_root.exists() else Path('.').resolve()
SRC = str(PROJECT_ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Windows only: winutils.exe needed by Hadoop filesystem layer
if os.name == 'nt':
    os.environ.setdefault('HADOOP_HOME', r'C:\hadoop')
    os.environ['PATH'] = os.environ['HADOOP_HOME'] + r'\bin;' + os.environ.get('PATH', '')

import pyspark
print(f'✅ Python      : {sys.version.split()[0]}')
print(f'✅ PySpark     : {pyspark.__version__}')
print(f'✅ NumPy       : {np.__version__}')
print(f'✅ Pandas      : {pd.__version__}')
print(f'✅ Project root: {PROJECT_ROOT}')

✅ Python      : 3.11.13
✅ PySpark     : 3.5.0
✅ NumPy       : 2.4.4
✅ Pandas      : 3.0.2
✅ Project root: /Users/varunraste/RAG_at_Scale/RAG_at_scale


---
## Section 1 — Spark Setup

**Key concepts:** Driver · Executors · Lazy evaluation · Catalyst optimizer

**Cluster tuning knobs to show students:**
| Config | Effect |
|---|---|
| `spark.driver.memory` | RAM for the driver process |
| `spark.executor.memory` | RAM per executor |
| `spark.sql.adaptive.enabled` | Auto-optimise join strategy at runtime |
| `spark.default.parallelism` | Default partitions for RDD ops |

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName('RAG_at_Scale_Demo')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

print(f'✅ Spark {spark.version}   UI → http://localhost:4040')

# Quick sanity check
df = spark.createDataFrame([('Alice', 25), ('Bob', 30), ('Charlie', 35)], ['name', 'age'])
df.show()
print(f'Rows: {df.count()}')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 15:09:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 3.5.0   UI → http://localhost:4040


+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



Rows: 3


In [3]:
# ── Lazy evaluation demo — nothing executes until .show() / .count() ──────
# Use the real raw docs for the demo (loaded from data/raw/*.txt)
raw_files = list((PROJECT_ROOT / 'data' / 'raw').glob('*.txt'))
demo_records = [(p.stem, p.read_text('utf-8').strip()) for p in raw_files]
demo_df = spark.createDataFrame(demo_records, ['doc_id', 'content'])

pipeline = (
    demo_df
    .withColumn('word_count', size(split(col('content'), ' ')))
    .withColumn('char_count', length(col('content')))
    .groupBy('doc_id')
    .agg(
        avg('word_count').alias('avg_words'),
        avg('char_count').alias('avg_chars')
    )
)

print('--- Logical plan (not yet executed) ---')
pipeline.explain()
print('--- Triggering execution now ---')
pipeline.show(truncate=False)

--- Logical plan (not yet executed) ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[doc_id#20], functions=[avg(word_count#24), avg(char_count#28)])
   +- Exchange hashpartitioning(doc_id#20, 200), ENSURE_REQUIREMENTS, [plan_id=59]
      +- HashAggregate(keys=[doc_id#20], functions=[partial_avg(word_count#24), partial_avg(char_count#28)])
         +- Project [doc_id#20, size(split(content#21,  , -1), true) AS word_count#24, length(content#21) AS char_count#28]
            +- Scan ExistingRDD[doc_id#20,content#21]


--- Triggering execution now ---


+------------+---------+---------+
|doc_id      |avg_words|avg_chars|
+------------+---------+---------+
|sample_doc_3|189.0    |1312.0   |
|sample_doc_2|149.0    |1120.0   |
|sample_doc_1|156.0    |1046.0   |
|sample_doc_5|209.0    |1692.0   |
|sample_doc_4|176.0    |1436.0   |
+------------+---------+---------+



---
## Section 2 — Document Processing

**Key concepts:** Native Spark functions vs Python UDFs · Chunking with overlap · Data quality

In [4]:
# Load the sample .txt files
raw_files = sorted((PROJECT_ROOT / 'data' / 'raw').glob('*.txt'))
print(f'Found {len(raw_files)} files')

records = [(p.stem, p.read_text('utf-8').strip()) for p in raw_files]
raw_df = spark.createDataFrame(records, ['doc_id', 'content'])
raw_df.show(truncate=80)

from IPython.display import Markdown, display

story_sections = [
    '## Story Corpus Preview',
    'This corpus intentionally mixes one real trend brief with fictional RAG incident reports so retrieval mistakes and reranking wins are easy to explain live.'
]
for doc_id, content in records:
    preview = content.replace('\n', ' ').strip()
    preview = preview[:260] + ('...' if len(preview) > 260 else '')
    story_sections.append(f"### `{doc_id}`\n{preview}")

display(Markdown('\n\n'.join(story_sections)))

Found 5 files


+------------+--------------------------------------------------------------------------------+
|      doc_id|                                                                         content|
+------------+--------------------------------------------------------------------------------+
|sample_doc_1|In 2026, the most significant shift in machine learning is the transition fro...|
|sample_doc_2|High above the digital skyline of the Obsidian-Archipelago, Senior Inference ...|
|sample_doc_3|Deep within the Silicon Valley server farms of Veridical Systems, Lead Data S...|
|sample_doc_4|Within the subterranean data-vaults of the Crystal-Reach research hub, Chief ...|
|sample_doc_5|Deep within the atmospheric-controlled chambers of the Chronos-Deep repositor...|
+------------+--------------------------------------------------------------------------------+



## Story Corpus Preview

This corpus intentionally mixes one real trend brief with fictional RAG incident reports so retrieval mistakes and reranking wins are easy to explain live.

### `sample_doc_1`
In 2026, the most significant shift in machine learning is the transition from passive models to Agentic AI and GraphRAG (Graph Retrieval-Augmented Generation). Unlike traditional LLMs that simply predict the next token, agentic systems are designed to autonom...

### `sample_doc_2`
High above the digital skyline of the Obsidian-Archipelago, Senior Inference Architect Lysandra Thorne monitored the pulse of the Eon-Registry, a massive decentralized database struggling with a phenomenon she dubbed the Labyrinthine-Drift. This unique form of...

### `sample_doc_3`
Deep within the Silicon Valley server farms of Veridical Systems, Lead Data Scientist Mia Thorne monitored the "Veritas-9" assistant as it began exhibiting signs of severe semantic hallucinations. The system, designed to provide legal advice, was erroneously p...

### `sample_doc_4`
Within the subterranean data-vaults of the Crystal-Reach research hub, Chief Algorithmic Officer Bastien Thorne confronted the total collapse of the Solstice-Core, a hyper-dimensional knowledge engine designed to synthesize centuries of planetary data. The cat...

### `sample_doc_5`
Deep within the atmospheric-controlled chambers of the Chronos-Deep repository, Kaelen Vance, the Principal Knowledge Architect, observed the volatile fluctuations of the Hyperion-Vector-Space. Beside him, Soren Krell, a Syntactic-Drift Engineer, frantically r...

In [5]:
# ── TEACHING POINT ─────────────────────────────────────────────────────────
# Use NATIVE Spark SQL functions whenever possible:
#   ✅ lower(), trim(), regexp_replace(), size(), split()  → JVM, no pickle
#   ⚠️  Python UDF                                         → serialised via cloudpickle
# ──────────────────────────────────────────────────────────────────────────
clean_df = (
    raw_df
    .withColumn('clean', trim(regexp_replace(lower(col('content')), r'\s+', ' ')))
    .withColumn('word_count', size(split(col('clean'), ' ')))
    .withColumn('sentence_count', size(split(col('clean'), r'\.\s+')))
)
clean_df.select('doc_id', 'word_count', 'sentence_count', 'clean').show(truncate=70)

+------------+----------+--------------+----------------------------------------------------------------------+
|      doc_id|word_count|sentence_count|                                                                 clean|
+------------+----------+--------------+----------------------------------------------------------------------+
|sample_doc_1|       156|             5|in 2026, the most significant shift in machine learning is the tran...|
|sample_doc_2|       149|             5|high above the digital skyline of the obsidian-archipelago, senior ...|
|sample_doc_3|       189|             7|deep within the silicon valley server farms of veridical systems, l...|
|sample_doc_4|       176|             6|within the subterranean data-vaults of the crystal-reach research h...|
|sample_doc_5|       210|             8|deep within the atmospheric-controlled chambers of the chronos-deep...|
+------------+----------+--------------+----------------------------------------------------------------

In [6]:
# ── Document chunking ────────────────────────────────────────────
# NOTE: Do NOT use max() here — pyspark.sql.functions.* shadows the builtin.
def chunk_words(text, chunk_size=80, overlap=15):
    if not text: return []
    words = text.split()
    step = chunk_size - overlap if chunk_size > overlap else 1   # avoids builtin max()
    return [" ".join(words[i:i+chunk_size])
            for i in range(0, len(words), step)
            if len(words[i:i+chunk_size]) >= chunk_size // 2]

# ── TEACHING POINT: production pattern would be a Pandas UDF ────────────────
#   chunk_udf = udf(chunk_words, ArrayType(StringType()))
#   chunks_df = (clean_df
#       .withColumn("chunks", chunk_udf("clean"))
#       .withColumn("chunk", explode("chunks")))
#
# For this demo (5 docs) we chunk on the driver to avoid Spark UDF worker
# socket instability under nbconvert / batch execution on Windows.
# ──────────────────────────────────────────────────────────────────────────
chunk_records = []
for row in clean_df.collect():
    for i, chunk in enumerate(chunk_words(row["clean"])):
        chunk_id = f"{row['doc_id']}_{i}"
        chunk_records.append((chunk_id, row["doc_id"], chunk))

from pyspark.sql.types import StructType, StructField, StringType as ST
chunk_schema = StructType([
    StructField("chunk_id", ST()),
    StructField("doc_id",   ST()),
    StructField("chunk",    ST()),
])
chunks_df = spark.createDataFrame(chunk_records, chunk_schema)
print(f"Total chunks: {chunks_df.count()}")
chunks_df.show(5, truncate=65)

Total chunks: 13
+--------------+------------+-----------------------------------------------------------------+
|      chunk_id|      doc_id|                                                            chunk|
+--------------+------------+-----------------------------------------------------------------+
|sample_doc_1_0|sample_doc_1|in 2026, the most significant shift in machine learning is the...|
|sample_doc_1_1|sample_doc_1|vector databases are integrated with knowledge graphs to provi...|
|sample_doc_2_0|sample_doc_2|high above the digital skyline of the obsidian-archipelago, se...|
|sample_doc_2_1|sample_doc_2|voltz attempted to stabilize the pipeline using the nebula-12 ...|
|sample_doc_3_0|sample_doc_3|deep within the silicon valley server farms of veridical syste...|
+--------------+------------+-----------------------------------------------------------------+
only showing top 5 rows



---
## Section 3 — Distributed Embeddings

**Key concepts:** Pandas UDF · Per-executor model caching · GPU scheduling · Batch size tuning

**GPU cluster config (production):**
```python
spark.config('spark.executor.resource.gpu.amount', '1')
spark.config('spark.task.resource.gpu.amount', '1')
spark.config('spark.executor.cores', '4')  # 4 CPU cores per GPU executor
```

In [7]:
import torch
gpu = torch.cuda.is_available()
device = 'cuda' if gpu else 'cpu'
print(f'GPU available : {gpu}')
if gpu:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')
else:
    print('  Using CPU — embeddings will be slower but functionally identical')
print(f'Device: {device}')

GPU available : False
  Using CPU — embeddings will be slower but functionally identical
Device: cpu


In [8]:
from embeddings.spark_embedder import SparkEmbedder

embedder = SparkEmbedder()          # uses all-MiniLM-L6-v2 (384-dim)

# Driver-side test
test_emb = embedder.embed_batch(['Machine learning is powerful.'])[0]
print(f'Model : {embedder.model_name}')
print(f'Dim   : {len(test_emb)}')
print(f'Sample: {test_emb[:5]}')

/Users/varunraste/RAG_at_Scale/RAG_at_scale/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9593.90it/s]

Model : sentence-transformers/all-MiniLM-L6-v2
Dim   : 384
Sample: [0.049713969230651855, -0.04058915376663208, 0.06119529530405998, 0.0031231865286827087, -0.027611924335360527]


In [9]:
# ── TEACHING POINT — Pandas UDF for distributed embedding ─────────────────
# In production (hundreds of GB of docs) you'd run this on a cluster:
#
#   embedded_df = chunks_df.withColumn('embedding', embedding_udf('chunk'))
#
# Key insight: capture only the model NAME (a string) in the closure.
# Each executor worker loads + caches the model once per process.
# ──────────────────────────────────────────────────────────────────────────
embedding_udf = embedder.get_embedding_udf()
print('Pandas UDF defined — model:', embedder.model_name)
print()

# ── Demo mode: embed on the driver (5 docs = no need for a cluster) ────────
# Collect text to driver → batched encode → rebuild as Spark DataFrame.
# Identical schema to the Pandas UDF path; all downstream cells just work.
n_total = chunks_df.count()
print(f'Embedding {n_total} chunks on driver ...')
t0 = time.time()

rows    = chunks_df.collect()
texts   = [r.chunk for r in rows]
vectors = embedder.embed_batch(texts)

elapsed = time.time() - t0
print(f'✅ {len(vectors)} chunks embedded in {elapsed:.1f}s  ({len(vectors)/elapsed:.1f} chunks/sec)')
print(f'   Embedding dim : {len(vectors[0])}')

# Rebuild as Spark DataFrame
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType
embed_schema = StructType([
    StructField('chunk_id',  StringType()),
    StructField('doc_id',    StringType()),
    StructField('chunk',     StringType()),
    StructField('embedding', ArrayType(FloatType())),
])
embed_records = [(r.chunk_id, r.doc_id, r.chunk, [float(x) for x in v])
                 for r, v in zip(rows, vectors)]
embedded_df = spark.createDataFrame(embed_records, embed_schema)
embedded_df.cache()
embedded_df.select('chunk_id', 'chunk', 'embedding').show(3, truncate=50)

Pandas UDF defined — model: sentence-transformers/all-MiniLM-L6-v2

Embedding 13 chunks on driver ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.08it/s]

✅ 13 chunks embedded in 0.2s  (71.6 chunks/sec)
   Embedding dim : 384


+--------------+--------------------------------------------------+--------------------------------------------------+
|      chunk_id|                                             chunk|                                         embedding|
+--------------+--------------------------------------------------+--------------------------------------------------+
|sample_doc_1_0|in 2026, the most significant shift in machine ...|[-0.03232724, -0.017732743, -0.010039166, 0.031...|
|sample_doc_1_1|vector databases are integrated with knowledge ...|[0.047749415, -0.071965866, 0.003916826, 0.0435...|
|sample_doc_2_0|high above the digital skyline of the obsidian-...|[-0.04835774, -8.610189E-4, -0.033994574, -0.06...|
+--------------+--------------------------------------------------+--------------------------------------------------+
only showing top 3 rows



In [10]:
EMBED_PATH = str(PROJECT_ROOT / 'data' / 'embeddings' / 'chunk_embeddings.parquet')
(PROJECT_ROOT / 'data' / 'embeddings').mkdir(parents=True, exist_ok=True)
embedded_df.write.mode('overwrite').parquet(EMBED_PATH)
print(f'💾 Embeddings saved to {EMBED_PATH}')

💾 Embeddings saved to /Users/varunraste/RAG_at_Scale/RAG_at_scale/data/embeddings/chunk_embeddings.parquet


---
## Section 4 — Hybrid Search (Dense + BM25)

**Key concepts:** Cosine similarity · BM25 · Min-max normalisation · Alpha fusion

```
final_score = α × dense_score + (1-α) × bm25_score
  α = 1.0 → pure semantic   |   α = 0.0 → pure keyword
```

In [11]:
from retrieval.hybrid_search import HybridSearch
from retrieval.reranker import Reranker

search_engine = HybridSearch()
reranker = Reranker()

# Load embeddings and build index
rows = spark.read.parquet(EMBED_PATH).collect()
docs       = [{'id': r.chunk_id, 'content': r.chunk} for r in rows]
embeddings = [r.embedding for r in rows]

search_engine.index_documents(docs, embeddings)
print(f'✅ Indexed {len(docs)} chunks')

✅ Indexed 13 chunks


In [12]:
# ── Alpha sweep — shows how fusion weight changes results ──────────────────
# Queries are grounded in the actual doc content (RAG/ML themed)
query = 'RAG pipeline hallucination and reranking'
q_emb = embedder.embed_batch([query])[0]

print(f'Query: "{query}"\n')
print(f'{"Mode":<22}  {"Score":>6}  Top chunk')
print('-' * 70)
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = search_engine.search(query, q_emb, top_k=3, alpha=alpha)
    top = results[0] if results else {}
    label = 'pure keyword' if alpha == 0 else ('pure semantic' if alpha == 1 else f'hybrid α={alpha}')
    snippet = next((d['content'] for d in docs if d['id'] == top.get('doc_id','')), '')[:60]
    print(f'{label:<22}  {top.get("score", 0):>6.3f}  {snippet}...')

Query: "RAG pipeline hallucination and reranking"

Mode                     Score  Top chunk
----------------------------------------------------------------------
pure keyword             1.000  the embedding model’s ability to distinguish between high-le...
hybrid α=0.3             0.798  voltz attempted to stabilize the pipeline using the nebula-1...
hybrid α=0.5             0.856  voltz attempted to stabilize the pipeline using the nebula-1...
hybrid α=0.7             0.913  voltz attempted to stabilize the pipeline using the nebula-1...
pure semantic            1.000  voltz attempted to stabilize the pipeline using the nebula-1...


---
## Section 5 — Cross-Encoder Reranking

**Why rerank?** Bi-encoder retrieval is fast but approximate. Cross-encoders see query + document jointly → better relevance, but slower. Use for top-K re-scoring only.

In [13]:
reranker.load_model()

query = 'cross-encoder reranking to fix hallucination in retrieval systems'
q_emb = embedder.embed_batch([query])[0]

# Retrieve more candidates, then rerank
candidates = search_engine.search(query, q_emb, top_k=10, alpha=0.6)
docs_for_rerank = [
    {'content': next((d['content'] for d in docs if d['id'] == r['doc_id']), ''),
     'doc_id': r['doc_id']}
    for r in candidates
]

reranked = reranker.rerank(query, docs_for_rerank, top_k=3)

print(f'Query: "{query}"\n')
print('After reranking:')
for i, d in enumerate(reranked, 1):
    print(f'  {i}. [{d["rerank_score"]:+.3f}] {d["content"][:90]}...')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10720.85it/s]

Query: "cross-encoder reranking to fix hallucination in retrieval systems"

After reranking:
  1. [+2.353] retrieval strategist, nadia vane, discovered that the system's aether-reranker was failing...
  2. [-0.185] the embedding model’s ability to distinguish between high-level stylistic similarities and...
  3. [-2.530] deep within the atmospheric-controlled chambers of the chronos-deep repository, kaelen van...


---
## Section 6 — FastAPI RAG Service

In [14]:
from service.app import app
from service.handlers import rag_service
from service.models import SearchRequest
from fastapi.testclient import TestClient

# Wire up the service with the indexed data
rag_service.initialize_search(docs, embeddings)
client = TestClient(app)

# Health
health = client.get('/health').json()
print('Health:', health['status'], '|', health['services'])

# Search
resp = client.post('/search', json={'query': 'machine learning', 'top_k': 3, 'rerank': True}).json()
print(f'\nQuery: {resp["query"]}')
print(f'Search: {resp["search_time"]*1000:.0f}ms  Rerank: {(resp.get("rerank_time") or 0)*1000:.0f}ms')
for r in resp['results']:
    score = r.get('rerank_score') or r['score']
    print(f'  [{score:.3f}] {r["content"][:70]}...')

{"asctime": "2026-05-02 15:09:46", "name": "observability.tracing", "levelname": "INFO", "message": "Tracing \u2192 console (no OTLP endpoint configured)"}


{"asctime": "2026-05-02 15:09:46", "name": "retrieval.hybrid_search", "levelname": "INFO", "message": "Dense index built (13 vectors, dim=384)"}


{"asctime": "2026-05-02 15:09:46", "name": "retrieval.hybrid_search", "levelname": "INFO", "message": "Indexed 13 documents (dense=True, sparse=\u2713)"}


{"asctime": "2026-05-02 15:09:46", "name": "service.handlers", "levelname": "INFO", "message": "Search engine initialised with 13 documents"}


{"asctime": "2026-05-02 15:09:46", "name": "httpx", "levelname": "INFO", "message": "HTTP Request: GET http://testserver/health \"HTTP/1.1 200 OK\""}


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.15it/s]

{"asctime": "2026-05-02 15:10:02", "name": "retrieval.reranker", "levelname": "INFO", "message": "Reranked 6 documents, returning top 3"}


{"asctime": "2026-05-02 15:10:02", "name": "observability.logging_config", "levelname": "INFO", "message": "Search Query Executed", "query_length": 16, "results_count": 3, "duration_seconds": 15.72800612449646}


{"asctime": "2026-05-02 15:10:02", "name": "httpx", "levelname": "INFO", "message": "HTTP Request: POST http://testserver/search \"HTTP/1.1 200 OK\""}



Query: machine learning
Search: 15728ms  Rerank: 9484ms
  [5.359] in 2026, the most significant shift in machine learning is the transit...
  [-5.954] vector databases are integrated with knowledge graphs to provide deep ...
  [-10.050] within the subterranean data-vaults of the crystal-reach research hub,...


In [15]:
print('To run the service as a live server (in a terminal):')
print('  uvicorn src.service.app:app --host 0.0.0.0 --port 8000 --reload')
print()
print('Then browse: http://localhost:8000/docs  (Swagger UI)')

To run the service as a live server (in a terminal):
  uvicorn src.service.app:app --host 0.0.0.0 --port 8000 --reload

Then browse: http://localhost:8000/docs  (Swagger UI)


---
## Section 7 — Observability: Logs · Traces · Metrics

**Stack:**
- **Structured logging** → JSON lines consumed by CloudWatch / ELK
- **OpenTelemetry** → traces to Jaeger / CloudWatch X-Ray
- **Prometheus** → metrics scraped every 15s, visualised in Grafana
- **Auto-scale trigger** → `rag_searches_performed_total` rate → ECS task count

In [16]:
from observability.logging_config import StructuredLogger, log_request, log_search_query

StructuredLogger.setup_logging(log_level='INFO', json_format=True)
logger = StructuredLogger.get_logger('demo')

logger.info('RAG demo started', extra={'section': 'observability', 'student': 'demo'})
log_request('req-001', 'POST', '/search', query_length=18)
log_search_query('machine learning', 3, 0.042)
print('✅ Structured JSON logs emitted — see console output above')

{"asctime": "2026-05-02 15:10:02", "name": "demo", "levelname": "INFO", "message": "RAG demo started", "section": "observability", "student": "demo"}


{"asctime": "2026-05-02 15:10:02", "name": "observability.logging_config", "levelname": "INFO", "message": "API Request", "request_id": "req-001", "method": "POST", "endpoint": "/search", "query_length": 18}


{"asctime": "2026-05-02 15:10:02", "name": "observability.logging_config", "levelname": "INFO", "message": "Search Query Executed", "query_length": 16, "results_count": 3, "duration_seconds": 0.042}


✅ Structured JSON logs emitted — see console output above


In [17]:
from opentelemetry import trace
from observability.tracing import tracing_manager, trace_search_query, trace_embedding_generation

tracing_manager.get_tracer()  # setup if needed, otherwise reuse the existing provider

with trace_search_query(query_length=20):
    time.sleep(0.04)

with trace_embedding_generation(doc_count=50):
    time.sleep(0.02)

trace.get_tracer_provider().force_flush()
time.sleep(0.2)

print('✅ Spans emitted to console in this section')
print('   In production → set OTLP_ENDPOINT to send to Jaeger or CloudWatch X-Ray')
print('   We flush here so later sections stay focused on retrieval and generation output.')

{
    "name": "search_query",
    "context": {
        "trace_id": "0x216e392e6dc1242620ad8e3701a34f85",
        "span_id": "0x9c5b80b3467370e3",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-05-02T09:40:02.481340Z",
    "end_time": "2026-05-02T09:40:02.526380Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query_length": 20
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.1",
            "service.name": "rag-service",
            "service.version": "1.0.0"
        },
        "schema_url": ""
    }
}
{
    "name": "embedding_generation",
    "context": {
        "trace_id": "0x7857e9cb4bc6969f3931e66983c6924f",
        "span_id": "0x79701da60e2bea9f",
        "trace_state": "[]"
    },
    "kind": "SpanKind

✅ Spans emitted to console in this section
   In production → set OTLP_ENDPOINT to send to Jaeger or CloudWatch X-Ray
   We flush here so later sections stay focused on retrieval and generation output.


In [18]:
import threading
from observability.metrics import metrics

# Start Prometheus endpoint in background
threading.Thread(target=lambda: metrics.start_server(port=8002), daemon=True).start()
time.sleep(0.3)

# Simulate some activity
metrics.searches_performed.inc(10)
metrics.embeddings_generated.inc(50)
metrics.index_size.set(len(docs))
with metrics.search_duration.time(): time.sleep(0.02)

import requests
raw = requests.get('http://localhost:8002/metrics').text
for line in raw.splitlines():
    if 'rag_' in line and not line.startswith('#'):
        print(line)

print('\n✅ Prometheus metrics live at http://localhost:8002/metrics')
print('   Start Grafana: docker-compose -f docker-compose-demo.yml --profile observability up')

{"asctime": "2026-05-02 15:10:02", "name": "observability.metrics", "levelname": "INFO", "message": "Prometheus metrics server started on port 8002"}


rag_embeddings_generated_total 50.0
rag_embeddings_generated_created 1.777714786640922e+09
rag_searches_performed_total 11.0
rag_searches_performed_created 1.777714786640927e+09
rag_embedding_duration_seconds_bucket{le="0.005"} 0.0
rag_embedding_duration_seconds_bucket{le="0.01"} 0.0
rag_embedding_duration_seconds_bucket{le="0.025"} 0.0
rag_embedding_duration_seconds_bucket{le="0.05"} 0.0
rag_embedding_duration_seconds_bucket{le="0.075"} 0.0
rag_embedding_duration_seconds_bucket{le="0.1"} 0.0
rag_embedding_duration_seconds_bucket{le="0.25"} 0.0
rag_embedding_duration_seconds_bucket{le="0.5"} 0.0
rag_embedding_duration_seconds_bucket{le="0.75"} 0.0
rag_embedding_duration_seconds_bucket{le="1.0"} 0.0
rag_embedding_duration_seconds_bucket{le="2.5"} 0.0
rag_embedding_duration_seconds_bucket{le="5.0"} 0.0
rag_embedding_duration_seconds_bucket{le="7.5"} 0.0
rag_embedding_duration_seconds_bucket{le="10.0"} 0.0
rag_embedding_duration_seconds_bucket{le="+Inf"} 0.0
rag_embedding_duration_seconds

---
## Section 7B — RAG Generation with AWS Bedrock (Claude)

This is the **G** in RAG. Retrieved chunks become the grounded context for the LLM.

In [19]:
import boto3, json as _json

bedrock = boto3.client('bedrock-runtime', region_name=os.environ.get('AWS_REGION', 'us-east-1'))

# Verify AWS credentials
try:
    sts = boto3.client('sts').get_caller_identity()
    print(f'✅ AWS Account: {sts["Account"]}   ARN: {sts["Arn"]}')
except Exception as e:
    print(f'⚠️  AWS not configured: {e}')
    print('   Run: aws configure')

{"asctime": "2026-05-02 15:10:03", "name": "botocore.credentials", "levelname": "INFO", "message": "Found credentials in shared credentials file: ~/.aws/credentials"}


✅ AWS Account: 593755927741   ARN: arn:aws:iam::593755927741:root


In [20]:
MODEL_ID = os.environ.get('BEDROCK_MODEL_ID', 'anthropic.claude-haiku-4-5-20251001-v1:0')
print(f'Using Bedrock model/profile: {MODEL_ID}')

def rag_answer(query: str, top_k: int = 3) -> str:
    q_emb = embedder.embed_batch([query])[0]
    hits = search_engine.search(query, q_emb, top_k=top_k * 2, alpha=0.6)
    pool = [{'content': next((d['content'] for d in docs if d['id'] == h['doc_id']), ''),
             'doc_id': h['doc_id']}
            for h in hits if any(d['id'] == h['doc_id'] for d in docs)]
    ranked = reranker.rerank(query, pool, top_k=top_k) if pool else []

    context = '\n\n'.join(f'[{i+1}] {d["content"]}' for i, d in enumerate(ranked)) or '(no context)'
    body = _json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 400,
        'system': 'Answer only from the provided context. Be concise.',
        'messages': [{'role': 'user',
                      'content': f'Context:\n{context}\n\nQuestion: {query}'}]
    })
    resp = bedrock.invoke_model(modelId=MODEL_ID, contentType='application/json',
                                accept='application/json', body=body)
    return _json.loads(resp['body'].read())['content'][0]['text']

print('These questions deliberately move across the story corpus: one real trend brief and two fictional RAG incidents.')

# Queries grounded in the actual doc content
for q in [
    'What is GraphRAG and how does it improve on standard RAG?',
    'In the Veritas-9 incident, why did the team add a cross-encoder before generation?',
    'What caused the Void-Leakage in the Chronos-Deep repository and how was it fixed?',
]:
    print(f'\n❓ {q}')
    try:
        print(f'🤖 {rag_answer(q)}')
    except Exception as e:
        msg = str(e)
        if 'inference profile' in msg.lower():
            print('⚠️  This account needs a Bedrock inference profile for the selected model.')
            print('   Set BEDROCK_MODEL_ID to your profile ID or ARN, then rerun this cell.')
        else:
            print(f'⚠️  {msg}')

Using Bedrock model/profile: us.anthropic.claude-sonnet-4-6
These questions deliberately move across the story corpus: one real trend brief and two fictional RAG incidents.

❓ What is GraphRAG and how does it improve on standard RAG?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.41it/s]

{"asctime": "2026-05-02 15:10:05", "name": "retrieval.reranker", "levelname": "INFO", "message": "Reranked 6 documents, returning top 3"}


🤖 ## GraphRAG (Graph Retrieval-Augmented Generation)

Based on the provided context, **GraphRAG** is an evolution of standard RAG that integrates **vector databases with knowledge graphs** to form a **"knowledge runtime."**

### How it improves on standard RAG:

| Standard RAG | GraphRAG |
|---|---|
| Simple retrieval from vector databases | Combines vector databases **with knowledge graphs** |
| Limited reasoning capability | Enables **deep reasoning** |
| Basic document retrieval | Provides **relationship-mapping** between concepts |

### Key Benefit:
GraphRAG supports the broader shift toward **agentic AI systems** that need to autonomously plan and execute multi-step workflows — capabilities that simple vector-based retrieval alone cannot adequately support.

> *"vector databases are integrated with knowledge graphs to provide deep reasoning and relationship-mapping that simple [retrieval cannot]"* — Context [1]

❓ In the Veritas-9 incident, why did the team add a cross-encoder bef

...

🤖 Based on the context, the team added a cross-encoder model before generation because **the lack of a robust reranking step meant the most relevant documents were being buried under noise**. While the vector database was successfully retrieving high-density clusters, the top-k results needed validation before reaching the generative large language model. The cross-encoder was implemented to rerank and validate those results, ensuring the most relevant documents surfaced rather than being obscured by less relevant ones.

❓ What caused the Void-Leakage in the Chronos-Deep repository and how was it fixed?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 26.97it/s]

{"asctime": "2026-05-02 15:10:13", "name": "retrieval.reranker", "levelname": "INFO", "message": "Reranked 6 documents, returning top 3"}


🤖 ## Cause of the Void-Leakage

Based on the context, the **void-leakage** was a rare catastrophic event where **historical embeddings from the antiquity-nodes began overwriting active production tokens**. This semantic interference caused the system's **omega-retrieval-layer** to hallucinate fragments of the **caelum-metadata** — a sensitive dataset governing navigational logic of regional transport networks.

## How It Was Fixed

The void-leakage was addressed through two key actions:

1. **Soren Krell** frantically recalibrated the **prism-encoder** to combat the void-leakage directly, while also subjecting affected elements to a **recursive entropy-purge**.

2. **Soren** also integrated a **contextual-anchor-protocol**, which forced the retriever to weight the **aether-index** based on **verifiable temporal markers** rather than pure cosine similarity.

These combined efforts stabilized the vector-drift, allowing the system to successfully isolate the **prime-key** — the singular g

---
## Section 8 — Docker & AWS Deploy

You are already inside Docker right now! The notebook is running in `jupyter/pyspark-notebook`.

### Production deployment flow
```
1. build image     → docker build -f docker/Dockerfile -t rag-service .
2. push to ECR     → bash scripts/deploy_aws.sh
3. rolling deploy  → aws ecs update-service --force-new-deployment
4. autoscale       → HPA on rag_searches_performed_total (Prometheus → ECS capacity provider)
```

In [21]:
import shutil
import subprocess

for cmd, label in [
    (['docker', '--version'], 'Docker'),
    (['aws', '--version'], 'AWS CLI'),
    (['aws', 'sts', 'get-caller-identity', '--query', 'Account', '--output', 'text'], 'AWS Account'),
]:
    binary = shutil.which(cmd[0])
    if not binary:
        print(f'❌ {label}: {cmd[0]} not installed')
        continue

    r = subprocess.run(cmd, capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'{status} {label}: {(r.stdout or r.stderr).strip()[:80]}')

print('\nDeployment commands:')
print('  bash scripts/deploy_aws.sh   # build → ECR push → ECS rolling deploy')

❌ Docker: docker not installed


✅ AWS CLI: aws-cli/1.45.2 Python/3.11.13 Darwin/25.4.0 botocore/1.43.2


✅ AWS Account: 593755927741

Deployment commands:
  bash scripts/deploy_aws.sh   # build → ECR push → ECS rolling deploy


---
## Section 9 — Autoscale with Kubernetes HPA

The [kubernetes/hpa.yaml](kubernetes/hpa.yaml) scales ECS tasks / K8s pods when search throughput rises.

**Cluster tuning recap for students:**

| Knob | Recommendation |
|---|---|
| Spark partitions | `2-4 × number of CPU cores` |
| Embedding batch size | `32` (CPU) / `128–256` (GPU) |
| UDF model cache | module-level dict, keyed by model name |
| HPA metric | `rag_searches_performed_total` rate |
| Docker memory limit | `driver_mem + executor_mem + 1g overhead` |

---
**🎓 Demo complete.** You ran the full RAG at Scale pipeline:

> **Spark** chunk → **Pandas UDF** embed → **Hybrid search** → **Cross-encoder** rerank → **FastAPI** serve → **Bedrock Claude** generate → **Prometheus/Grafana** observe → **ECR/ECS** deploy